# Audio: full-scale reset-corrected check

Self-contained notebook that settles the one open question for the audio section of the paper. It retrains the small audio Transformer on the 60-recording `waves_yesno` bundle (48 train / 12 held-out, ~8 min on a T4), then computes, **from this single run**:

1. real and shuffled held-out losses,
2. the uncorrected harmonic sweep (periods 255/k),
3. the reset-corrected test at every candidate pitch period (62.0, 63.75, 64.0, 64.2, 64.5, 67.2 samples) for real **and** shuffled data,
4. autocorrelation and YIN pitch estimates on the same 12 held-out recordings.

**Before running:** `Runtime > Change runtime type > T4 GPU`. When asked, upload `audio_bundle_full60.zip` (from the repository). Then `Runtime > Run all`. Four files download automatically at the end; also copy the printed block under *Summary*.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('Warning: GPU is not enabled!')
torch.manual_seed(11)
np.random.seed(11)


device: cuda


In [2]:
from google.colab import files
uploaded = files.upload()  # select audio_bundle_full60.zip
!unzip -o audio_bundle_full60.zip -d /content/data > /dev/null
print('full-scale audio ready (60 recordings, 12 held out)')

Saving audio_bundle_full60.zip to audio_bundle_full60.zip
full-scale audio ready (60 recordings, 12 held out)


## Architecture
Same small GPT as in the other notebooks (4 layers, 4 heads, 128 dims, 256-position context).

In [4]:
VOCAB_SIZE = 256
BLOCK_SIZE = 256
N_LAYER = 4
N_HEAD = 4
N_EMBD = 128
DROPOUT = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(N_EMBD, 3*N_EMBD)
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.n_head = N_HEAD
        self.drop = nn.Dropout(DROPOUT)
        mask = torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(N_EMBD, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (hd ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(
            nn.Linear(N_EMBD, 4*N_EMBD), nn.GELU(),
            nn.Linear(4*N_EMBD, N_EMBD), nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Parameter(torch.zeros(1, BLOCK_SIZE, N_EMBD))
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1), reduction='mean')
        return logits, loss

def get_batch(data_split, batch_size):
    ix = torch.randint(0, len(data_split) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([data_split[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

def train_model(train_data, steps=6000, lr=3e-4, batch_size=64):
    model = TinyGPT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    t0 = time.time()
    model.train()
    for step in range(steps):
        xb, yb = get_batch(train_data, batch_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 200 == 0 or step == steps - 1:
            print(f'step {step:5d}  loss {loss.item():.4f}  bits/byte {loss.item()/0.6931:.4f}  '
                  f'({time.time()-t0:.0f}s)')
    print('training done:', round(time.time()-t0), 's')
    return model

def extract_losses(model, heldout_data):
    model.eval()
    n_chunks = len(heldout_data) // BLOCK_SIZE
    per_position_bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)
    with torch.no_grad():
        for c in range(n_chunks):
            chunk = heldout_data[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
            x = chunk[:-1].unsqueeze(0).to(device)
            y = chunk[1:].unsqueeze(0).to(device)
            logits, _ = model(x)
            logp = F.log_softmax(logits, dim=-1)
            token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
            bits = (-token_logp / np.log(2)).cpu().numpy()
            start = c*(BLOCK_SIZE-1)
            per_position_bits[start:start+(BLOCK_SIZE-1)] = bits
    print('mean bits/byte on held-out:', per_position_bits.mean())
    return per_position_bits, n_chunks

## Audio data (`waves_yesno`)
Quantized to 256 levels; the first 12 recordings (sorted by name) are held out entirely.

In [5]:
import wave

AUDIO_DIR = '/content/data/multimodal_data/audio'
files_audio = sorted(os.listdir(AUDIO_DIR))
n_heldout_audio = max(1, len(files_audio)//5)
heldout_audio_files = files_audio[:n_heldout_audio]
train_audio_files = files_audio[n_heldout_audio:]
print('train audio files:', len(train_audio_files), ' heldout:', len(heldout_audio_files))

def load_and_quantize(file_list):
    arrs = []
    for f in file_list:
        w = wave.open(os.path.join(AUDIO_DIR, f), 'rb')
        d = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
        q = ((d.astype(int) + 32768) * 256 // 65536).clip(0, 255)
        arrs.append(q)
    return np.concatenate(arrs)

audio_train_bytes = load_and_quantize(train_audio_files)
audio_heldout_bytes = load_and_quantize(heldout_audio_files)
print('train samples:', len(audio_train_bytes), ' heldout samples:', len(audio_heldout_bytes))

audio_train_data = torch.tensor(audio_train_bytes, dtype=torch.long)
audio_heldout_data = torch.tensor(audio_heldout_bytes, dtype=torch.long)

train audio files: 48  heldout: 12
train samples: 2337920  heldout samples: 603440


## Train

In [6]:
audio_model = train_model(audio_train_data, steps=6000)

step     0  loss 6.0002  bits/byte 8.6571  (1s)
step   200  loss 1.3847  bits/byte 1.9978  (16s)
step   400  loss 1.1706  bits/byte 1.6889  (30s)
step   600  loss 1.0904  bits/byte 1.5732  (45s)
step   800  loss 1.1591  bits/byte 1.6724  (60s)
step  1000  loss 1.0984  bits/byte 1.5848  (76s)
step  1200  loss 1.0215  bits/byte 1.4739  (91s)
step  1400  loss 1.2143  bits/byte 1.7519  (107s)
step  1600  loss 0.9244  bits/byte 1.3337  (123s)
step  1800  loss 1.0587  bits/byte 1.5275  (140s)
step  2000  loss 0.9324  bits/byte 1.3452  (156s)
step  2200  loss 0.9152  bits/byte 1.3205  (172s)
step  2400  loss 0.8291  bits/byte 1.1962  (188s)
step  2600  loss 0.9869  bits/byte 1.4240  (204s)
step  2800  loss 0.8310  bits/byte 1.1990  (220s)
step  3000  loss 0.8975  bits/byte 1.2949  (236s)
step  3200  loss 0.8963  bits/byte 1.2932  (252s)
step  3400  loss 0.8534  bits/byte 1.2313  (268s)
step  3600  loss 0.9075  bits/byte 1.3094  (284s)
step  3800  loss 1.0300  bits/byte 1.4861  (300s)
step  40

## Held-out losses (real and shuffled)
The shuffled control permutes the held-out samples with a fixed seed and re-evaluates the same trained model.

In [7]:
audio_losses, audio_n_chunks = extract_losses(audio_model, audio_heldout_data)

g = torch.Generator().manual_seed(123)
audio_heldout_shuf = audio_heldout_data[torch.randperm(len(audio_heldout_data), generator=g)]
audio_shuf_losses, _ = extract_losses(audio_model, audio_heldout_shuf)

np.save('audio_losses_60_rerun.npy', audio_losses)
np.save('audio_shuf_losses_60_rerun.npy', audio_shuf_losses)
np.save('audio_meta_60_rerun.npy', np.array([BLOCK_SIZE, audio_n_chunks, N_LAYER, N_HEAD, N_EMBD,
                                             sum(p.numel() for p in audio_model.parameters())]))
print('saved; windows:', audio_n_chunks, ' predictions per window:', BLOCK_SIZE-1)

mean bits/byte on held-out: 1.3210638
mean bits/byte on held-out: 5.222696
saved; windows: 2357  predictions per window: 255


## Pitch estimates on the same held-out recordings (independent of the model)

In [8]:
!pip install librosa -q
import librosa
FS = 8000

def autocorr_pitch(y, sr, fmin=70, fmax=280, frame_len=1024, hop=512):
    f0s = []
    for start in range(0, len(y) - frame_len, hop):
        frame = y[start:start+frame_len].astype(float)
        frame = frame - frame.mean()
        if np.abs(frame).max() < 50:
            continue
        corr = np.correlate(frame, frame, mode='full')
        corr = corr[len(corr)//2:]
        min_lag, max_lag = int(sr/fmax), int(sr/fmin)
        if max_lag >= len(corr):
            continue
        lag = np.argmax(corr[min_lag:max_lag]) + min_lag
        if corr[lag] <= 0.3 * corr[0]:
            continue
        f0s.append(sr / lag)
    return np.array(f0s)

autocorr_f0, yin_f0 = [], []
for f in heldout_audio_files:
    w = wave.open(os.path.join(AUDIO_DIR, f), 'rb')
    d = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    autocorr_f0.append(autocorr_pitch(d, FS))
    yv = librosa.yin(d.astype(float)/32768.0, fmin=70, fmax=280, sr=FS, frame_length=1024)
    yin_f0.append(yv[yv > 0])
autocorr_hz = float(np.median(np.concatenate(autocorr_f0)))
yin_hz = float(np.median(np.concatenate(yin_f0)))
print('autocorrelation median F0:', round(autocorr_hz, 1), 'Hz  (period', round(FS/autocorr_hz, 2), 'samples)')
print('YIN median F0:', round(yin_hz, 1), 'Hz  (period', round(FS/yin_hz, 2), 'samples)')

autocorrelation median F0: 125.0 Hz  (period 64.0 samples)
YIN median F0: 119.1 Hz  (period 67.16 samples)


## Analysis and summary

In [9]:
out = []
def P(*a):
    s = ' '.join(str(x) for x in a); print(s); out.append(s)

W = BLOCK_SIZE - 1   # context-reset period (255 predictions per window)

def z_at(sig, period):
    s = sig - sig.mean()
    sp = np.abs(np.fft.rfft(s)); f = np.fft.rfftfreq(len(s))
    i = np.argmin(np.abs(f - 1.0/period))
    others = np.delete(sp[1:], i-1)
    return (sp[i] - others.mean()) / others.std()

def band_peak(sig, lo=40, hi=200):
    s = sig - sig.mean()
    sp = np.abs(np.fft.rfft(s)); f = np.fft.rfftfreq(len(s))
    m = (f > 1.0/hi) & (f < 1.0/lo)
    return 1.0 / f[m][np.argmax(sp[m])]

def resid(losses, W=W):
    L = losses[:(len(losses)//W)*W].reshape(-1, W)
    return (L - L.mean(axis=0, keepdims=True)).ravel()

P('=== SUMMARY (copy everything from here) ===')
P('device:', device, '| windows:', audio_n_chunks, '| N =', len(audio_losses))
P(f'mean bits/byte  real: {audio_losses.mean():.4f}   shuffled: {audio_shuf_losses.mean():.4f}')
P()
P('--- independent pitch estimates (12 held-out recordings) ---')
P(f'autocorrelation: {autocorr_hz:.1f} Hz (period {FS/autocorr_hz:.2f})   YIN: {yin_hz:.1f} Hz (period {FS/yin_hz:.2f})')
P()
P('--- uncorrected: harmonics of the reset period, z real vs shuffled ---')
P(f'{"k":>3} {"period":>8} {"z real":>8} {"z shuf":>8}')
for k in [2, 3, 4, 5, 6]:
    per = W / k
    P(f'{k:>3} {per:>8.2f} {z_at(audio_losses, per):>8.2f} {z_at(audio_shuf_losses, per):>8.2f}')
P('band (40-200) peak, uncorrected: real period', round(band_peak(audio_losses), 2),
  '| shuffled period', round(band_peak(audio_shuf_losses), 2))
P()
P('--- reset-corrected (position-mean removed) ---')
rr, rs = resid(audio_losses), resid(audio_shuf_losses)
P('band (40-200) peak after correction: real period', round(band_peak(rr), 2),
  '(', round(FS/band_peak(rr), 1), 'Hz )  | shuffled period', round(band_peak(rs), 2))
P(f'{"period":>8} {"z real":>8} {"z shuf":>8}')
for per in [62.0, 63.75, 64.0, 64.2, 64.5, 67.2]:
    P(f'{per:>8.2f} {z_at(rr, per):>8.2f} {z_at(rs, per):>8.2f}')
P('=== END SUMMARY ===')

with open('audio_check_results.txt', 'w') as fh:
    fh.write('\n'.join(out))

=== SUMMARY (copy everything from here) ===
device: cuda | windows: 2357 | N = 601035
mean bits/byte  real: 1.3211   shuffled: 5.2227

--- independent pitch estimates (12 held-out recordings) ---
autocorrelation: 125.0 Hz (period 64.00)   YIN: 119.1 Hz (period 67.16)

--- uncorrected: harmonics of the reset period, z real vs shuffled ---
  k   period   z real   z shuf
  2   127.50     7.25    34.54
  3    85.00     5.40    28.67
  4    63.75     8.95    22.61
  5    51.00     4.28    19.10
  6    42.50     3.92    20.10
band (40-200) peak, uncorrected: real period 63.75 | shuffled period 127.5

--- reset-corrected (position-mean removed) ---
band (40-200) peak after correction: real period 43.87 ( 182.3 Hz )  | shuffled period 151.93
  period   z real   z shuf
   62.00     1.48    -0.86
   63.75    -0.86    -1.79
   64.00     0.38    -0.95
   64.20     0.99     0.59
   64.50     1.63    -1.12
   67.20     0.31     1.22
=== END SUMMARY ===


## Download results
Sends back the losses (real and shuffled), the meta file, and the printed summary.

In [10]:
from google.colab import files
for fn in ['audio_losses_60_rerun.npy', 'audio_shuf_losses_60_rerun.npy',
           'audio_meta_60_rerun.npy', 'audio_check_results.txt']:
    files.download(fn)
print('4 files downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4 files downloaded
